<a href="https://colab.research.google.com/github/Manikanta-23-ai/LANGCHAIN-LEARNING/blob/main/LangChain_Expression_Language.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install langchain_core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 574.7/574.7 kB 7.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.6/883.6 kB 11.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 29.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [langchain_core]


In [3]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [4]:
prompt = ChatPromptTemplate.from_template("""
Answer the question using only the context provided.
If the answer is not present, say you could not find it in the PDF.

Context:
{context}

Question:
{question}

Answer:
""")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip install -q langchain-chroma langchain-huggingface sentence-transformers

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

DB_DIR = "/content/drive/MyDrive/langchain_project/chroma_db"

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = Chroma(
    collection_name="my_pdf",
    embedding_function=embedding_model,
    persist_directory=DB_DIR
)

retriever = db.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever loaded successfully!")

Mounted at /content/drive


Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


In [ ]:
docs = retriever.invoke("What is the main topic of this PDF?")

for doc in docs:
    print("Page:", doc.metadata.get("page", "Unknown"))
    print(doc.page_content[:300])
    print("-" * 40)

In [ ]:
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer using only the context below.
If the answer is missing, say you could not find it in the PDF.

Context:
{context}

Question:
{question}

Answer:
""")

def prepare_inputs(question):
    docs = retriever.invoke(question)

    return {
        "question": question,
        "context": "\n\n".join(
            doc.page_content for doc in docs
        )
    }

def generate_answer(inputs):
    full_prompt = prompt.invoke(inputs).to_string()

    response = llm(
        full_prompt,
        max_new_tokens=150,
        do_sample=False
    )

    return response[0]["generated_text"]

rag_chain = (
    RunnableLambda(prepare_inputs)
    | RunnableLambda(generate_answer)
)

print(rag_chain.invoke("What is the main topic of this PDF?"))